# Hetebølgen i Europa juni 2026
## Varmekuppel, rekorder og klimaendringer

I siste halvdel av juni 2026 lå et kraftig høytrykk over Vest-Europa i nesten to uker. Mange land satte nye varmerekorder for juni, og noen satte rekord for hele året. Ifølge Copernicus var juni 2026 den varmeste junimåneden som er målt i Vest-Europa. Midt i juli kom varmen også til Sør-Norge.

I denne økten bruker du ekte data til å svare på hovedspørsmålet nedenfor.

> ☀️ **Hovedspørsmål**
> Hvorfor ble det så varmt i juni 2026, og hvor mye av varmen kan vi knytte til klimaendringer?

| Tid | Del |
|---|---|
| 0–10 min | Oppstart og hypotese |
| 10–30 min | Del 1 Hvor varmt ble det? |
| 30–55 min | Del 2 Varmekuppelen |
| 55–65 min | Pause |
| 65–90 min | Del 3 Er dette klimaendringer? |
| 90–105 min | Del 4 Hva med Norge? |
| 105–120 min | Del 5 Oppsummering |

**Slik jobber du**
- Kjør cellene i rekkefølge med Shift + Enter.
- Svar i cellene merket ✏️. Dobbeltklikk for å skrive, og trykk Shift + Enter når du er ferdig.
- I cellene merket 🔧 skal du endre en verdi selv og kjøre cellen på nytt.
- Har du tid til overs, finner du to fordypningsoppgaver helt til slutt.

### Oppstart
Hva husker du fra sommeren 2026? Skriv en hypotese om hva som må til for at det blir hetebølge, og om du tror klimaendringer spilte en rolle.

✏️ **Svar**

*Min hypotese er at …*

### Last inn verktøy og data
Kjør cellen nedenfor. Den henter inn Python-pakkene og leser datafilene i mappen `data`. Les meldingene som skrives ut.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from matplotlib.colors import TwoSlopeNorm
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature

DATA = Path("data")
PC = ccrs.PlateCarree()           # koordinatene i dataene er lengde- og breddegrad
PROJ = ccrs.LambertConformal(central_longitude=10, standard_parallels=(35, 65))
EUROPA = [-15, 32, 35, 66]        # V, Ø, S, N

plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "axes.grid": True, "grid.alpha": 0.3})

BYER = {"Madrid": (40.42, -3.70), "Paris": (48.85, 2.35), "London": (51.51, -0.13),
        "Berlin": (52.52, 13.40), "Oslo": (59.94, 10.72), "Nesbyen": (60.57, 9.11),
        "Bergen": (60.38, 5.33)}
# Fargeblindvennlige farger (Okabe–Ito) og ulike markører
FARGER = ["#E69F00", "#56B4E9", "#009E73", "#0072B2", "#D55E00", "#CC79A7", "#000000"]
MERKER = ["o", "s", "^", "D", "v", "P", "X"]


def komma(ax, x=False, y=True):
    """Desimalkomma og ekte minus på aksene."""
    f = mticker.FuncFormatter(lambda v, p: f"{v:g}".replace(".", ",").replace("-", "−"))
    if x:
        ax.xaxis.set_major_formatter(f)
    if y:
        ax.yaxis.set_major_formatter(f)


def datoakse(ax, dager=5):
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=dager))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m"))


def kart(ax, utsnitt=EUROPA):
    """Kartbakgrunn med kystlinje, landegrenser og gradnett med norske retninger."""
    ax.set_extent(utsnitt, crs=PC)
    ax.coastlines(resolution="50m", lw=0.6, color="0.2", zorder=3)
    ax.add_feature(cfeature.BORDERS, lw=0.3, edgecolor="0.4", zorder=3)
    gl = ax.gridlines(draw_labels=True, x_inline=False, y_inline=False,
                      lw=0.4, color="0.5", ls=":")
    gl.top_labels = gl.right_labels = False
    gl.xformatter = mticker.FuncFormatter(
        lambda v, p=None: "0°" if v == 0 else f"{abs(v):.0f}° {'V' if v < 0 else 'Ø'}")
    gl.yformatter = mticker.FuncFormatter(lambda v, p=None: f"{v:.0f}° N")


def merk_byer(ax, byer=BYER, farge="k"):
    for navn, (la, lo) in byer.items():
        ax.plot(lo, la, "o", ms=3, color=farge, transform=PC, zorder=5)
        ax.text(lo + 0.6, la + 0.3, navn, fontsize=7, color=farge, transform=PC, zorder=5)


def punkt(felt, by):
    la, lo = BYER[by]
    return felt.sel(lat=la, lon=lo, method="nearest")


def tall(v, des=1):
    return f"{v:.{des}f}".replace(".", ",").replace("-", "−")


def finnes(fil):
    if (DATA / fil).exists():
        return True
    print(f"⚠️  Finner ikke {fil} i mappen data. Spør læreren.")
    return False


def les(fil):
    return xr.open_dataset(DATA / fil) if finnes(fil) else None


tmaks = les("era5_tmaks.nc")
tmiddel = les("era5_tmiddel.nc")
luft = les("era5_trykkflater.nc")
klima_t = les("era5_klima_t2m.nc")
klima_z = les("era5_klima_z500.nc")

serier = {}
for f in sorted(DATA.glob("punkt_*.csv")):
    navn = next(b for b in BYER if b.lower() == f.stem.replace("punkt_", ""))
    serier[navn] = pd.read_csv(f, parse_dates=["dato"]).set_index("dato")
stasjoner = {f.stem.replace("stasjon_", "").capitalize():
             pd.read_csv(f, parse_dates=["dato"]).set_index("dato")
             for f in sorted(DATA.glob("stasjon_*.csv"))}
print("Punktserier fra ERA5:", ", ".join(serier) or "ingen")
print("Norske stasjoner:", ", ".join(stasjoner) or "ingen")
print("Klar ✅")

---
## Del 1 Hvor varmt ble det? (20 min)

### 1a Høyeste temperatur
Kartet viser den høyeste temperaturen 2 meter over bakken i hver rute i ERA5 i perioden du velger. Svarte linjer viser 35 °C og 40 °C.

> 📘 **Begrep** **ERA5** er en **reanalyse**. Det betyr at en værvarslingsmodell har regnet seg fram til været time for time, styrt av alle målinger som finnes. Resultatet er et komplett datasett for hele jorden fra 1940 til i dag, med ruter på omtrent 25 × 25 km.

In [ ]:
START, SLUTT = "2026-06-18", "2026-06-30"     # 🔧

if tmaks is not None:
    tx = tmaks.tmaks.sel(time=slice(START, SLUTT)).max("time")
    fig = plt.figure(figsize=(8, 6.5))
    ax = plt.axes(projection=PROJ)
    kart(ax)
    cf = ax.contourf(tx.lon, tx.lat, tx, levels=np.arange(16, 46, 2), cmap="YlOrRd",
                     extend="both", transform=PC)
    cs = ax.contour(tx.lon, tx.lat, tx, levels=[35, 40], colors="k",
                    linewidths=[0.8, 1.6], transform=PC)
    ax.clabel(cs, fmt="%d °C", fontsize=8)
    merk_byer(ax)
    fig.colorbar(cf, ax=ax, shrink=0.75, label="Høyeste temperatur (°C)")
    ax.set_title(f"Høyeste temperatur {START[8:]}.{START[5:7]} til {SLUTT[8:]}.{SLUTT[5:7]}.2026")
    fig.text(0.02, 0.02, "Kilde: ERA5, Copernicus/ECMWF. Lambert konform kjegleprojeksjon.",
             fontsize=7)
    plt.show()

    print("Høyeste verdi i hver by i perioden")
    for by in BYER:
        s = punkt(tmaks.tmaks, by).sel(time=slice(START, SLUTT))
        print(f"  {by:8s} {tall(float(s.max()))} °C den {pd.Timestamp(s.idxmax().values):%d.%m}")

**Oppgaver**
1. Hvor i Europa var det varmest? Hvor langt nord nådde 35 °C-linjen?
2. Hvilken by fikk høyest temperatur, og hvilken dag?
3. ERA5 viser gjennomsnittet for en rute på 25 × 25 km. Tror du den høyeste temperaturen målt på en værstasjon ble høyere eller lavere enn ERA5? Begrunn svaret.

✏️ **Svar**

*Skriv svaret ditt her.*

### 1b Dag for dag

In [ ]:
VALGTE = ["Madrid", "Paris", "London", "Berlin", "Oslo"]     # 🔧

if tmaks is not None:
    fig, ax = plt.subplots(figsize=(10, 4.2), constrained_layout=True)
    ax.axvspan(pd.Timestamp(START), pd.Timestamp(SLUTT), color="0.9", zorder=0)
    for i, by in enumerate(VALGTE):
        s = punkt(tmaks.tmaks, by)
        ax.plot(s.time, s, color=FARGER[i], marker=MERKER[i], ms=3, lw=1.3, label=by)
    ax.set_ylabel("Høyeste temperatur (°C)")
    ax.set_title("Høyeste temperatur hvert døgn, juni og juli 2026 (grått felt viser perioden fra 1a)")
    ax.legend(ncol=len(VALGTE), loc="upper left", fontsize=8)
    datoakse(ax); komma(ax)
    plt.show()

**Oppgaver**
1. Når var det varmest i de ulike byene? Kom toppen samtidig overalt?
2. Hvor mange dager på rad hadde Paris over 30 °C?
3. Finner du tegn til den neste hetebølgen i juli?

✏️ **Svar**

*Skriv svaret ditt her.*

### 1c Hvor uvanlig var det?

> 📘 **Begrep** Et **avvik** er forskjellen mellom en målt verdi og en referanseverdi. Et annet ord for det samme er **anomali**. Her er referansen gjennomsnittet for juni i normalperioden 1991–2020. Et avvik på +5 °C betyr at det var 5 °C varmere enn normalt.

In [ ]:
if tmiddel is not None and klima_t is not None:
    snitt = tmiddel.tmiddel.sel(time=slice(START, SLUTT)).mean("time")
    mnd = pd.Timestamp(START).month
    avvik = snitt - klima_t.t2m.sel(mnd=mnd).interp(lat=snitt.lat, lon=snitt.lon)

    fig = plt.figure(figsize=(8, 6.5))
    ax = plt.axes(projection=PROJ)
    kart(ax)
    cf = ax.contourf(avvik.lon, avvik.lat, avvik, levels=np.arange(-8, 9, 1),
                     cmap="RdBu_r", norm=TwoSlopeNorm(0), extend="both", transform=PC)
    cs = ax.contour(avvik.lon, avvik.lat, avvik, levels=[4, 6], colors="k",
                    linewidths=0.7, transform=PC)
    ax.clabel(cs, fmt="+%d °C", fontsize=8)
    merk_byer(ax)
    fig.colorbar(cf, ax=ax, shrink=0.75, label="Avvik i døgnmiddeltemperatur (°C)")
    ax.set_title("Avvik fra normalen (1991–2020) i perioden fra 1a")
    fig.text(0.02, 0.02, "Kilde: ERA5, Copernicus/ECMWF", fontsize=7)
    plt.show()

    for by in BYER:
        v = float(punkt(avvik, by))
        print(f"  {by:8s} avvik {'+' if v > 0 else ''}{tall(v)} °C")

**Oppgaver**
1. Hvor var avviket størst? Er det samme sted som der det var varmest i 1a? Forklar forskjellen.
2. Var det noen deler av Europa som var kaldere enn normalt?

✏️ **Svar**

*Skriv svaret ditt her.*

---
## Del 2 Varmekuppelen (25 min)

> 📘 **Begrep** En **varmekuppel** er et kraftig og stillestående høytrykk i høyden. Luften synker langsomt i høytrykket og blir varmet opp når den blir presset sammen. Synkende luft gir også klar himmel, så solen varmer bakken dag etter dag. Når høytrykket blir liggende i ro og stenger ute lavtrykkene fra vest, kaller vi det en **blokkering**.

> 📘 **Begrep** **Geopotensiell høyde** for 500 hPa forteller hvor høyt over havet lufttrykket er 500 hPa, oppgitt i dekameter (1 gpdam ≈ 10 m). Varm luft tar større plass enn kald luft, så 500 hPa-flaten ligger høyere over varme luftmasser. Normalt ligger den rundt 552–576 gpdam om sommeren.

### 2a Høytrykket i høyden
Linjene viser høyden til 500 hPa-flaten. Fargene viser avviket fra normalen for måneden.

In [ ]:
DATOER = ["2026-06-16", "2026-06-22", "2026-06-28"]     # 🔧

if luft is not None and klima_z is not None:
    fig, akser = plt.subplots(1, 3, figsize=(14, 4.8), subplot_kw={"projection": PROJ},
                              constrained_layout=True)
    for ax, d in zip(akser, DATOER):
        kart(ax, [-35, 40, 30, 72])
        z = luft.z500.sel(time=d)
        avvik = z - klima_z.z500.sel(mnd=pd.Timestamp(d).month).interp(lat=z.lat, lon=z.lon)
        cf = ax.contourf(z.lon, z.lat, avvik, levels=np.arange(-30, 31, 5), cmap="RdBu_r",
                         norm=TwoSlopeNorm(0), extend="both", transform=PC)
        cs = ax.contour(z.lon, z.lat, z, levels=np.arange(520, 610, 6), colors="k",
                        linewidths=0.8, transform=PC)
        ax.clabel(cs, fmt="%d", fontsize=7)
        m = avvik.where(avvik == avvik.max(), drop=True)
        ax.text(float(m.lon[0]), float(m.lat[0]), "H", fontsize=14, weight="bold",
                color="k", ha="center", va="center", transform=PC, zorder=6)
        ax.set_title(f"{pd.Timestamp(d):%d.%m.%Y}")
    fig.colorbar(cf, ax=akser, shrink=0.7, label="Avvik i geopotensiell høyde 500 hPa (gpdam)")
    fig.text(0.01, 0.0, "Kilde: ERA5, Copernicus/ECMWF. Linjer viser 500 hPa-flaten hver 6. gpdam. "
             "H er der avviket er størst.", fontsize=7)
    plt.show()

**Oppgaver**
1. Hvor lå høytrykket i høyden de tre dagene? Flyttet det seg mye?
2. Se på formen på linjene rundt høytrykket. Ser du lavtrykk i høyden (tråg) på begge sider? Et slikt mønster kalles en omegablokkering fordi det ligner den greske bokstaven Ω.
3. Hvordan påvirker en blokkering lavtrykkene som vanligvis kommer inn fra Atlanterhavet?

✏️ **Svar**

*Skriv svaret ditt her.*

### 2b Varm luft i 850 hPa
Temperaturen i 850 hPa, omtrent 1,5 km over havet, påvirkes lite av døgnet og bakken. Den viser hvilken luftmasse som ligger over et område.

In [ ]:
if luft is not None:
    fig, akser = plt.subplots(1, 3, figsize=(14, 4.8), subplot_kw={"projection": PROJ},
                              constrained_layout=True)
    for ax, d in zip(akser, DATOER):
        kart(ax, [-35, 40, 30, 72])
        t = luft.t850.sel(time=d)
        cf = ax.contourf(t.lon, t.lat, t, levels=np.arange(-4, 30, 2), cmap="YlOrRd",
                         extend="both", transform=PC)
        cs = ax.contour(t.lon, t.lat, t, levels=[20, 24], colors="k", linewidths=0.8,
                        transform=PC)
        ax.clabel(cs, fmt="%d °C", fontsize=7)
        merk_byer(ax, {b: BYER[b] for b in ["Madrid", "Paris", "Berlin", "Oslo"]})
        ax.set_title(f"{pd.Timestamp(d):%d.%m.%Y}")
    fig.colorbar(cf, ax=akser, shrink=0.7, label="Temperatur i 850 hPa (°C)")
    fig.text(0.01, 0.0, "Kilde: ERA5, Copernicus/ECMWF", fontsize=7)
    plt.show()

**Oppgaver**
1. Hvor kommer den varmeste luften fra? Tenk på hvor den varme luften ligger den første dagen og hvor den ligger senere.
2. Hvor langt nord nådde 20 °C i 850 hPa?

✏️ **Svar**

*Skriv svaret ditt her.*

### 2c Fra 1,5 km høyde til bakken
Når luft synker, blir den presset sammen og varmes opp. Tørr luft blir omtrent **1 °C varmere per 100 meter** den synker (tørradiabatisk temperaturgradient). Cellen nedenfor sammenligner temperaturen i 850 hPa med høyeste temperatur ved bakken.

In [ ]:
BY = "Paris"            # 🔧
HOYDE_M = 1500          # 🔧 omtrentlig høyde til 850 hPa-flaten

if luft is not None and tmaks is not None:
    t850 = punkt(luft.t850, BY).to_series()
    z500 = punkt(luft.z500, BY).to_series()
    tx = punkt(tmaks.tmaks, BY).to_series().reindex(t850.index)
    tsynk = t850 + 9.8 * HOYDE_M / 1000            # luften ført tørradiabatisk ned

    fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True, constrained_layout=True)
    a1.plot(z500.index, z500, color="0.2", marker="o", ms=3)
    a1.set_ylabel("500 hPa-flaten (gpdam)")
    a1.set_title(f"{BY}, juni og juli 2026")
    a2.plot(tx.index, tx, color="#D55E00", marker="o", ms=3, label="Høyeste temperatur ved bakken")
    a2.plot(t850.index, t850, color="#0072B2", marker="s", ms=3, label="Temperatur i 850 hPa")
    a2.plot(tsynk.index, tsynk, color="#0072B2", ls="--", label="850 hPa-luft ført ned til bakken")
    a2.set_ylabel("Temperatur (°C)")
    a2.legend(fontsize=8, loc="upper left")
    datoakse(a2); komma(a1); komma(a2)
    plt.show()

    r = np.corrcoef(t850.values, tx.values)[0, 1]
    print(f"Korrelasjon mellom T850 og høyeste temperatur ved bakken: {tall(r, 2)}")

**Oppgaver**
1. Beskriv hvordan 500 hPa-flaten, temperaturen i 850 hPa og temperaturen ved bakken henger sammen.
2. Stemmer den stiplede kurven godt med temperaturen ved bakken på de varmeste dagene? Hva betyr det for forklaringen av varmen?
3. Nevn to andre ting enn synkende luft som gjør at bakken blir ekstra varm i en varmekuppel. Tenk på skydekke og på hvor tørr jorden er. Hvorfor blir det varmere når jorden er tørr?

✏️ **Svar**

*Skriv svaret ditt her.*

---
## ☕ Pause (10 min)

---
## Del 3 Er dette klimaendringer? (25 min)

Ett enkelt værfenomen kan ikke i seg selv bevise klimaendringer. Men vi kan se om slike hendelser er blitt vanligere og sterkere over tid. Her bruker du ERA5-data for én by fra 1940 til 2026.

### 3a Årets varmeste dag

In [ ]:
BY = "Paris"      # 🔧 velg mellom byene som ble skrevet ut da du lastet data

s = serier[BY]
txx = s.tmaks.groupby(s.index.year).max()
txx = txx[txx.index <= 2026]
glatt = txx.rolling(30, center=True, min_periods=20).mean()

fig, ax = plt.subplots(figsize=(10, 4.2), constrained_layout=True)
ax.plot(txx.index, txx, "o", color="#D55E00", ms=4, label="Varmeste dag i året")
ax.plot(glatt.index, glatt, color="k", lw=2, label="Glidende 30-årsmiddel")
for aar in [1976, 2003, 2019, 2026]:
    if aar in txx.index:
        ax.annotate(str(aar), (aar, txx[aar]), xytext=(0, 8), textcoords="offset points",
                    ha="center", fontsize=8)
ax.set_xlabel("År")
ax.set_ylabel("Høyeste temperatur (°C)")
ax.set_title(f"Årets varmeste dag i {BY}, 1940–2026")
ax.legend(loc="upper left")
komma(ax)
fig.text(0.01, 0.0, "Kilde: ERA5 via Open-Meteo", fontsize=7)
plt.show()

print(f"Gjennomsnitt 1951–1980: {tall(txx.loc[1951:1980].mean())} °C")
print(f"Gjennomsnitt 1996–2025: {tall(txx.loc[1996:2025].mean())} °C")
print(f"2026: {tall(txx.loc[2026])} °C")

**Oppgaver**
1. Hvor mye varmere er årets varmeste dag i gjennomsnitt nå enn i perioden 1951–1980?
2. Hvordan var 2026 sammenlignet med 1976 og 2003?
3. Prøv en annen by. Er mønsteret det samme?

✏️ **Svar**

*Skriv svaret ditt her.*

### 3b Hvor mange varme dager?

In [ ]:
TERSKEL = 30      # 🔧 °C

dager = (s.tmaks >= TERSKEL).groupby(s.index.year).sum()
dager = dager[dager.index <= 2026]
perioder = {"1961–1990": (1961, 1990), "1991–2020": (1991, 2020), "2021–2026": (2021, 2026)}

fig, ax = plt.subplots(figsize=(10, 3.8), constrained_layout=True)
ax.bar(dager.index, dager.values, color="#E69F00", edgecolor="k", lw=0.3)
for navn, (a, b) in perioder.items():
    m = dager.loc[a:b].mean()
    ax.hlines(m, a - 0.5, b + 0.5, color="k", lw=2)
    ax.text((a + b) / 2, m + 0.5, tall(m), ha="center", fontsize=8)
ax.set_xlabel("År")
ax.set_ylabel(f"Dager med minst {TERSKEL} °C")
ax.set_title(f"{BY}, dager per år med høyeste temperatur minst {TERSKEL} °C")
fig.text(0.01, 0.0, "Kilde: ERA5 via Open-Meteo. Svarte streker viser gjennomsnittet for periodene.",
         fontsize=7)
plt.show()

**Oppgaver**
1. Hvordan har antallet varme dager endret seg fra 1961–1990 til 2021–2026?
2. Velg en høyere terskel, for eksempel 35 °C. Blir endringen større eller mindre i prosent? Hvorfor?

✏️ **Svar**

*Skriv svaret ditt her.*

### 3c Terningene er lastet
Når gjennomsnittet øker litt, blir de ekstreme verdiene mye vanligere. Figuren nedenfor sammenligner fordelingen av høyeste temperatur i juni i to perioder.

In [ ]:
juni = s.tmaks[s.index.month == 6]
gammel = juni[(juni.index.year >= 1951) & (juni.index.year <= 1980)]
ny = juni[(juni.index.year >= 1996) & (juni.index.year <= 2025)]
topp26 = juni[juni.index.year == 2026].max()

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
kanter = np.arange(np.floor(juni.min()), np.ceil(max(juni.max(), topp26)) + 1, 1)
ax.hist(gammel, bins=kanter, density=True, histtype="step", lw=2, color="#0072B2",
        label="Juni 1951–1980")
ax.hist(ny, bins=kanter, density=True, histtype="stepfilled", alpha=0.5, color="#D55E00",
        edgecolor="#D55E00", hatch="//", label="Juni 1996–2025")
ax.axvline(topp26, color="k", ls="--", lw=1.5)
ax.text(topp26, ax.get_ylim()[1] * 0.9, " juni 2026", fontsize=8)
ax.set_xlabel("Høyeste temperatur (°C)")
ax.set_ylabel("Andel av dagene")
ax.set_title(f"Fordeling av høyeste temperatur i juni, {BY}")
ax.legend()
komma(ax, x=True)
plt.show()

for navn, d in [("1951–1980", gammel), ("1996–2025", ny)]:
    print(f"{navn}  middel {tall(d.mean())} °C, "
          f"dager over {tall(topp26 - 3, 0)} °C: {(d > topp26 - 3).sum()}")

> 📘 **Begrep** **Attribusjonsstudier** sammenligner sannsynligheten for en hendelse i dagens klima med et klima uten menneskeskapt oppvarming. World Weather Attribution (WWA) fant at hetebølgen 18.–29. juni 2026 ville vært omtrent 3,5 °C kaldere på dagtid i klimaet fra 1976, og omtrent 2 °C kaldere i klimaet fra 2003. De skrev at varmen ville vært så godt som umulig i juni for 50 år siden. Selve værmønsteret var ikke spesielt uvanlig.

**Oppgaver**
1. Beskriv hvordan fordelingen har flyttet seg. Hvor stor er forskjellen i middelverdi?
2. Hvorfor blir det mange flere dager over en høy terskel selv om middelverdien bare har økt litt?
3. WWA skrev at værmønsteret ikke var spesielt uvanlig. Hva betyr det for hvordan vi skal forstå hetebølgen?

✏️ **Svar**

*Skriv svaret ditt her.*

---
## Del 4 Hva med Norge? (15 min)

> 📘 **Begrep** Meteorologisk institutt definerer en **norsk hetebølge** som fem eller flere sammenhengende dager der høyeste temperatur er 27 °C eller mer. Tidligere brukte MET en dansk definisjon der gjennomsnittet av høyeste temperatur over tre dager måtte være minst 28 °C.

### 4a Hetebølger i Oslo
Cellen bruker stasjonsdata fra Blindern hvis læreren har lastet dem ned, og ellers ERA5 for Oslo.

In [ ]:
TERSKEL_N = 27      # 🔧 °C
DAGER_N = 5         # 🔧 sammenhengende dager


def finn_hetebolger(tmaks, terskel, dager):
    """Finner perioder med minst `dager` dager på rad med tmaks >= terskel."""
    varm = (tmaks >= terskel).astype(int)
    gruppe = (varm.diff() != 0).cumsum()
    ut = []
    for _, g in varm.groupby(gruppe):
        if g.iloc[0] == 1 and len(g) >= dager:
            ut.append((g.index[0], g.index[-1], len(g)))
    df = pd.DataFrame(ut, columns=["start", "slutt", "dager"])
    df["start"] = pd.to_datetime(df["start"]); df["slutt"] = pd.to_datetime(df["slutt"])
    return df


if "Blindern" in stasjoner:
    oslo, kilde = stasjoner["Blindern"].tmaks.dropna(), "Oslo – Blindern (MET Norge)"
else:
    oslo, kilde = serier["Oslo"].tmaks, "Oslo (ERA5 via Open-Meteo)"
oslo = oslo.asfreq("D")

hb = finn_hetebolger(oslo, TERSKEL_N, DAGER_N)
per_aar = hb.groupby(hb.start.dt.year).size().reindex(
    range(oslo.index.year.min(), 2027), fill_value=0)

fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6.5), constrained_layout=True)
a1.bar(per_aar.index, per_aar.values, color="#D55E00", edgecolor="k", lw=0.3)
a1.set_xlabel("År"); a1.set_ylabel("Antall hetebølger")
a1.set_title(f"Hetebølger per år, {kilde}")

sommer = oslo.loc["2026-06-01":"2026-08-31"]
a2.plot(sommer.index, sommer, color="#D55E00", marker="o", ms=3)
a2.axhline(TERSKEL_N, color="k", ls="--", lw=1)
for _, r in hb[hb.start.dt.year == 2026].iterrows():
    a2.axvspan(r.start, r.slutt, color="#E69F00", alpha=0.3)
a2.set_ylabel("Høyeste temperatur (°C)")
a2.set_title("Sommeren 2026 (oransje felt viser hetebølger)")
datoakse(a2, 7); komma(a2)
plt.show()

print(f"Hetebølger 1961–1990: {per_aar.loc[1961:1990].sum()}, "
      f"1991–2020: {per_aar.loc[1991:2020].sum()}")
print("Hetebølger i 2026:")
print(hb[hb.start.dt.year == 2026].to_string(index=False) if (hb.start.dt.year == 2026).any()
      else "  ingen")

**Oppgaver**
1. Hvor mange hetebølger var det i Oslo i 1961–1990 og i 1991–2020?
2. Fikk Oslo en hetebølge sommeren 2026 etter MET sin definisjon?
3. Endre definisjonen til `TERSKEL_N = 25`. Hvordan endrer resultatet seg? Hvorfor er det viktig å oppgi definisjonen når man snakker om hetebølger?

✏️ **Svar**

*Skriv svaret ditt her.*

### 4b Nesbyen i juli 2026
Den høyeste temperaturen i Norge i juli 2026 var 34,2 °C på Nesbyen–Todokk den 15. juli (MET Norge). Den norske rekorden er 35,6 °C fra Nesbyen i 1970.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8), constrained_layout=True)
e = serier["Nesbyen"].tmaks.loc["2026-07-01":"2026-07-31"]
ax.plot(e.index, e, color="#0072B2", marker="s", ms=3, label="ERA5 (rute på 25 × 25 km)")
if "Todokk" in stasjoner:
    st = stasjoner["Todokk"].tmaks.loc["2026-07-01":"2026-07-31"]
    ax.plot(st.index, st, color="#D55E00", marker="o", ms=3, label="Nesbyen–Todokk (stasjon)")
ax.axhline(27, color="k", ls="--", lw=1)
ax.set_ylabel("Høyeste temperatur (°C)")
ax.set_title("Nesbyen i juli 2026")
ax.legend(fontsize=8)
datoakse(ax, 3); komma(ax)
plt.show()
print(f"Høyeste verdi i ERA5: {tall(e.max())} °C")

**Oppgaver**
1. Hvor stor er forskjellen mellom ERA5 og stasjonen på den varmeste dagen?
2. Nesbyen ligger i bunnen av Hallingdal. Forklar hvorfor en dalbunn kan bli varmere enn gjennomsnittet for en rute på 25 × 25 km som også dekker fjell.

✏️ **Svar**

*Skriv svaret ditt her.*

---
## Del 5 Oppsummering (15 min)

Gå tilbake til hovedspørsmålet og hypotesen din fra starten.

> ☀️ Hvorfor ble det så varmt i juni 2026, og hvor mye av varmen kan vi knytte til klimaendringer?

Skriv et svar på 150–250 ord. Bruk minst **tre konkrete tall** fra figurene ovenfor, og vis til hvilken figur tallene kommer fra. Skill tydelig mellom **været** (varmekuppelen) og **klimaet** (de lange seriene). Skriv også om hypotesen din stemte.

✏️ **Svar**

*Hetebølgen i juni 2026 skyldtes …*

**Refleksjon**
Du har brukt både reanalyse (ERA5) og målinger fra værstasjoner. Hva er fordelene og ulempene med hver av dem? Tenk på hvor langt tilbake dataene går, hvor de finnes, og hvor nøyaktige de er på ett bestemt sted.

✏️ **Svar**

*Skriv svaret ditt her.*

---
## Fordypning A Marin hetebølge

Copernicus meldte om marine hetebølger i det vestlige Middelhavet og langs atlanterhavskysten i juni 2026. Varmt hav gir varme og fuktige netter langs kysten.

In [ ]:
if finnes("oisst_avvik.nc"):
    sa = xr.open_dataset(DATA / "oisst_avvik.nc").anom
    snitt = sa.sel(time=slice(START, SLUTT)).mean("time")
    fig = plt.figure(figsize=(8, 6))
    ax = plt.axes(projection=PROJ)
    kart(ax, [-20, 38, 32, 62])
    cf = ax.contourf(snitt.lon, snitt.lat, snitt, levels=np.arange(-4, 4.5, 0.5),
                     cmap="RdBu_r", norm=TwoSlopeNorm(0), extend="both", transform=PC)
    ax.add_feature(cfeature.LAND, facecolor="0.85", zorder=2)
    fig.colorbar(cf, ax=ax, shrink=0.75, label="Avvik i havtemperatur (°C)")
    ax.set_title("Avvik i havtemperatur fra 1991–2020 i perioden fra 1a")
    fig.text(0.02, 0.02, "Kilde: NOAA OISST v2.1", fontsize=7)
    plt.show()

    med = sa.sel(lat=slice(36, 44), lon=slice(-2, 10)).mean(["lat", "lon"])
    fig, ax = plt.subplots(figsize=(10, 3), constrained_layout=True)
    ax.plot(med.time, med, color="#D55E00")
    ax.axhline(0, color="k", lw=0.8)
    ax.set_ylabel("Avvik (°C)")
    ax.set_title("Vestlige Middelhavet (36–44° N, 2° V–10° Ø)")
    datoakse(ax, 7); komma(ax)
    plt.show()

**Oppgaver**
1. Hvor var avviket i havtemperatur størst?
2. Forklar hvorfor et varmt hav gjør nettene varmere langs kysten.

✏️ **Svar**

*Skriv svaret ditt her.*

## Fordypning B Tropenetter

En **tropenatt** er en natt der laveste temperatur er minst 20 °C. Varme netter er farlige fordi kroppen ikke får hvilt seg fra varmen.

In [ ]:
BY_B = "Madrid"     # 🔧

s2 = serier[BY_B]
netter = (s2.tmin >= 20).groupby(s2.index.year).sum()
netter = netter[netter.index <= 2026]
fig, ax = plt.subplots(figsize=(10, 3.8), constrained_layout=True)
ax.bar(netter.index, netter.values, color="#0072B2", edgecolor="k", lw=0.3)
ax.set_xlabel("År"); ax.set_ylabel("Tropenetter")
ax.set_title(f"Tropenetter per år i {BY_B}")
fig.text(0.01, 0.0, "Kilde: ERA5 via Open-Meteo", fontsize=7)
plt.show()
print(f"1961–1990: {tall(netter.loc[1961:1990].mean())} per år, "
      f"1991–2020: {tall(netter.loc[1991:2020].mean())} per år, 2026: {netter.loc[2026]}")

**Oppgaver**
1. Hvordan har antallet tropenetter endret seg?
2. Sammenlign en kystby og en innlandsby. Hvor er det flest tropenetter, og hvorfor?

✏️ **Svar**

*Skriv svaret ditt her.*